## Final Project Submission

Please fill out:
* Student name: 
* Student pace: self paced / part time / full time
* Scheduled project review date/time: 
* Instructor name: 
* Blog post URL:


# Movie Studio Analysis

## Business Understanding

The goal of this project is to analyze movie industry data and provide actionable recommendations to a company planning to create a new movie studio.

We will analyze movie characteristics, ratings, genres, runtime, and box office performance to identify factors associated with successful movies.

## Data Understanding

The analysis will use movie industry datasets containing information about movies, ratings, genres, runtime, and box office performance.

The main datasets we will work with are:

- IMDb movie data stored in `im.db`
- Box Office Mojo data stored in `bom.movie_gross.csv`
- Additional movie datasets available in the `zippedData` folder

We will first inspect the available data, understand the columns and data types, and then determine which datasets are most useful for answering our business questions.

In [62]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

### Load the Box Office Mojo Dataset

We load the Box Office Mojo dataset into a pandas DataFrame so that we can inspect its structure and assess its data quality before cleaning.

In [66]:
# Load the Box Office Mojo movie gross dataset
bom = pd.read_csv("zippedData/bom.movie_gross.csv")

### Preview the Dataset

We display the first few records to understand the type of information contained in the dataset.

In [68]:
# Display the first five rows of the dataset
bom.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


### Dataset Dimensions

We check the number of rows and columns in the Box Office Mojo dataset to understand its size before carrying out further inspection and cleaning.

In [69]:
# Check the number of rows and columns
bom.shape

(3387, 5)

### Dataset Structure

We inspect the dataset structure to identify the column names, number of non-null observations, and data types. This helps us identify columns that may require cleaning.

In [70]:
# Display column names, non-null counts, and data types
bom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB


### Missing Values

We check each column for missing values to identify incomplete records and determine which variables require attention during the cleaning stage.

In [71]:
# Count missing values in each column
bom.isna().sum()

title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

### Duplicate Records

We check for completely duplicated rows to ensure that duplicate movie records do not cause movies or revenue to be counted more than once.

In [72]:
# Count completely duplicated rows
bom.duplicated().sum()

0

### Inspecting Box Office Revenue

The `domestic_gross` and `foreign_gross` columns contain box-office revenue. We inspect their values and data types before deciding how they should be cleaned and prepared for analysis.

In [73]:
# Inspect the data types of the revenue columns
bom[["domestic_gross", "foreign_gross"]].dtypes

domestic_gross    float64
foreign_gross      object
dtype: object

In [74]:
# Display sample revenue values
bom[["domestic_gross", "foreign_gross"]].head(10)

,domestic_gross,foreign_gross
0,415000000.0,652000000
1,334200000.0,691300000
2,296000000.0,664300000
3,292600000.0,535700000
4,238700000.0,513900000
5,300500000.0,398000000
6,312400000.0,311500000
7,200800000.0,391000000
8,251500000.0,291600000
9,217600000.0,277300000


## BOM Dataset Inspection Findings

The Box Office Mojo dataset contains **3,387 movie records and 5 variables**: `title`, `studio`, `domestic_gross`, `foreign_gross`, and `year`.

The inspection shows that `title` and `year` have complete records, while `studio`, `domestic_gross`, and `foreign_gross` contain missing values.

- `studio` has **5 missing values**.
- `domestic_gross` has **28 missing values**.
- `foreign_gross` has **1,350 missing values**.

The data types also reveal an important issue. `domestic_gross` is stored as a numeric (`float64`) variable, while `foreign_gross` is stored as an `object`. Therefore, `foreign_gross` will require further cleaning and conversion before it can be reliably used in numerical calculations.

The duplicate check returned **0 completely duplicated rows**, indicating that no identical records need to be removed from the dataset.

Overall, the inspection identifies missing values and an inconsistent data type in `foreign_gross` as the main data-quality issues to address during the cleaning stage.

## Data Preparation

Based on the inspection findings, the BOM dataset requires cleaning before it can be used for analysis.

The cleaning process will focus on:

1. Handling missing values in `studio`, `domestic_gross`, and `foreign_gross`.
2. Converting `foreign_gross` from an object to a numeric variable.
3. Checking the cleaned values for consistency.
4. Preparing the revenue variables for calculating total box-office performance.
5. Preserving valid movie records while avoiding unnecessary data loss.

## Data Cleaning

### Handling Missing Studio Values

The `studio` column identifies the movie's production or distribution studio.

Our inspection identified 5 records with missing studio values. Before deciding how to handle these records, we inspect them to determine whether the studio information can be recovered from the available data or whether the records should be excluded from studio-level analysis.

In [75]:
# Display movies with missing studio values
bom[bom["studio"].isna()]

,title,studio,domestic_gross,foreign_gross,year
210,Outside the Law (Hors-la-loi),NaN,96900.0,3300000,2010
555,Fireflies in the Garden,NaN,70600.0,3300000,2011
933,Keith Lemon: The Film,NaN,NaN,4000000,2012
1862,Plot for Peace,NaN,7100.0,NaN,2014
2825,Secret Superstar,NaN,NaN,122000000,2017


In [76]:
# Display the title, gross revenue, and year for movies with missing studios
bom.loc[
    bom["studio"].isna(),
    ["title", "domestic_gross", "foreign_gross", "year"]
]

,title,domestic_gross,foreign_gross,year
210,Outside the Law (Hors-la-loi),96900.0,3300000,2010
555,Fireflies in the Garden,70600.0,3300000,2011
933,Keith Lemon: The Film,NaN,4000000,2012
1862,Plot for Peace,7100.0,NaN,2014
2825,Secret Superstar,NaN,122000000,2017


### Handling Missing Studio Values

The inspection identified 5 movie records with missing values in the `studio` column.

The affected records contain valid movie information and should not be removed simply because the studio is unavailable. Since the available BOM data does not provide enough information to reliably determine the missing studios, we will replace the missing studio values with `"Unknown"`.

This approach preserves the movie records while clearly identifying that the studio information was unavailable. The `"Unknown"` category can be excluded from studio-level comparisons where appropriate.

In [77]:
# Replace missing studio values with "Unknown"
bom["studio"] = bom["studio"].fillna("Unknown")

### Verify Studio Cleaning

We verify that all missing values in the `studio` column have been handled successfully.

In [78]:
# Check the number of remaining missing studio values
bom["studio"].isna().sum()

0

In [81]:
# Confirm that five records were assigned to the "Unknown" category
bom["studio"].value_counts().get("Unknown", 0)

5

In [80]:
# Confirm that the number of rows remains unchanged after cleaning
bom.shape

(3387, 5)

### Studio Cleaning Result

The 5 missing values in the `studio` column were replaced with `"Unknown"`.

No movie records were removed during this process, so the dataset remains at 3,387 rows. Using an `"Unknown"` category preserves valid movie records while clearly indicating that the studio information was unavailable.

These `"Unknown"` records will be considered appropriately when performing studio-level analysis.